# Aula 17 — Cross-validation: estimando generalização sem desperdiçar dados

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/03-machine-learning/notebooks/17-cross-validation-laboratorio.ipynb)

**Pergunta:** quanto a estimativa de desempenho muda quando os folds representam registros aleatórios, entidades novas ou o futuro?

Hipóteses pré-registradas:

1. em registros repetidos, StratifiedKFold será otimista porque a mesma entidade atravessa treino e validação;
2. GroupKFold eliminará toda sobreposição e estimará generalização para entidades novas;
3. TimeSeriesSplit com gap 2 nunca usará futuro e manterá duas posições de embargo;
4. num problema iid separado, previsões OOF e teste externo produzirão estimativas compatíveis, sem serem idênticas.


## Ambiente e dependências

- Python ≥ 3.10
- NumPy ≥ 1.24
- Matplotlib ≥ 3.7
- scikit-learn ≥ 1.3

Os dados são sintéticos e gerados localmente. A seed é fixa; não há download, segredo ou credencial.


In [ ]:
import platform
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.metrics import (
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import (
    GroupKFold,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("error")
SEED = 20260908
rng = np.random.default_rng(SEED)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("Seed:", SEED)

## 1. Cálculo manual: média e dispersão

Reproduzimos o exemplo da aula. O desvio usa ddof igual a 1, isto é, denominador \(K-1\). Ele descreve os folds observados; não é tratado como erro-padrão iid.


In [ ]:
scores_exemplo = np.array([0.71, 0.76, 0.74, 0.62, 0.77])
media_exemplo = scores_exemplo.mean()
desvio_exemplo = scores_exemplo.std(ddof=1)

print(f"Média: {media_exemplo:.6f}")
print(f"Desvio amostral: {desvio_exemplo:.6f}")
assert np.isclose(media_exemplo, 0.72)
assert np.isclose(desvio_exemplo, np.sqrt(0.0146 / 4))

## 2. Entidades repetidas

Geramos 180 entidades, cinco registros por entidade. Cada entidade possui uma assinatura de oito dimensões praticamente constante e um rótulo fixo. Existe também um sinal populacional fraco. A unidade de análise é o registro; a unidade de generalização desejada é a entidade.


In [ ]:
n_groups, repeats, fingerprint_dim = 180, 5, 8
groups = np.repeat(np.arange(n_groups), repeats)
group_labels = np.tile([0, 1], n_groups // 2)
rng.shuffle(group_labels)
y_grouped = np.repeat(group_labels, repeats)

fingerprints = rng.normal(size=(n_groups, fingerprint_dim))
X_fingerprint = np.repeat(fingerprints, repeats, axis=0)
X_fingerprint += rng.normal(0, 0.025, size=X_fingerprint.shape)
weak_signal = y_grouped + rng.normal(0, 1.8, size=len(y_grouped))
X_grouped = np.column_stack([X_fingerprint, weak_signal])

print("X:", X_grouped.shape, "y:", y_grouped.shape)
print("Entidades:", np.unique(groups).size, "prevalência:", f"{y_grouped.mean():.6f}")
assert X_grouped.shape == (900, 9)
assert np.all(np.bincount(groups) == repeats)
assert y_grouped.mean() == 0.5

### 2.1 Dois contratos de split

O mesmo pipeline KNN é avaliado com dois contratos:

- StratifiedKFold: novos registros, permitindo entidades já vistas;
- GroupKFold: entidades inteiramente novas.

Usamos balanced accuracy e exatamente o mesmo dataset. A comparação revela a pergunta respondida pelo splitter, não uma propriedade absoluta do algoritmo.


In [ ]:
model_grouped = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=3, weights="distance"),
)
cv_random = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_groups = GroupKFold(n_splits=5)

random_res = cross_validate(
    model_grouped, X_grouped, y_grouped, cv=cv_random,
    scoring="balanced_accuracy", return_indices=True,
)
group_res = cross_validate(
    model_grouped, X_grouped, y_grouped, groups=groups, cv=cv_groups,
    scoring="balanced_accuracy", return_indices=True,
)

random_scores = random_res["test_score"]
group_scores = group_res["test_score"]
print("StratifiedKFold:", np.round(random_scores, 6),
      f"média={random_scores.mean():.6f}")
print("GroupKFold:     ", np.round(group_scores, 6),
      f"média={group_scores.mean():.6f}")
assert random_scores.mean() > group_scores.mean() + 0.20
assert random_scores.mean() > 0.90
assert 0.40 < group_scores.mean() < 0.75

### 2.2 Auditoria de sobreposição

Contamos quantas entidades aparecem nos dois lados de cada fold. O splitter aleatório deve vazar muitas identidades; o splitter por grupo deve produzir zero em todas as rodadas.


In [ ]:
def overlaps_by_fold(splitter, X, y, group_ids, pass_groups):
    iterator = splitter.split(X, y, group_ids) if pass_groups else splitter.split(X, y)
    overlaps = []
    for train_idx, val_idx in iterator:
        overlap = np.intersect1d(group_ids[train_idx], group_ids[val_idx])
        overlaps.append(len(overlap))
        if pass_groups:
            assert len(overlap) == 0
    return np.array(overlaps)

overlap_random = overlaps_by_fold(cv_random, X_grouped, y_grouped, groups, False)
overlap_group = overlaps_by_fold(cv_groups, X_grouped, y_grouped, groups, True)

print("Entidades sobrepostas — aleatório:", overlap_random)
print("Entidades sobrepostas — por grupo:", overlap_group)
assert np.all(overlap_random > 100)
assert np.all(overlap_group == 0)

### 2.3 Visualização e interpretação

Cada ponto é um fold. A linha tracejada em 0,5 representa chance balanceada. A distância entre contratos mede o otimismo produzido por validar registros de entidades conhecidas quando o deploy exige entidades novas.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x1 = np.arange(1, 6) - 0.08
x2 = np.arange(1, 6) + 0.08
ax.scatter(x1, random_scores, s=65, label="StratifiedKFold: registros")
ax.scatter(x2, group_scores, s=65, label="GroupKFold: entidades")
ax.axhline(0.5, color="black", linestyle="--", linewidth=1, label="chance")
ax.set(xticks=np.arange(1, 6), xlabel="Fold", ylabel="Balanced accuracy",
       title="Mesmo modelo, perguntas de generalização diferentes")
ax.set_ylim(0.35, 1.03)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

otimismo = random_scores.mean() - group_scores.mean()
print(f"Otimismo do split de registros: {otimismo:.6f}")

## 3. Validação temporal com embargo

Criamos 60 posições ordenadas. Com quatro splits, horizonte de 10 e gap 2, todo treino deve terminar antes da validação e exatamente duas posições intermediárias devem ficar excluídas.


In [ ]:
time_index = np.arange(60)
temporal_cv = TimeSeriesSplit(n_splits=4, test_size=10, gap=2)
temporal_rows = []

for fold, (train_idx, val_idx) in enumerate(temporal_cv.split(time_index), start=1):
    observed_gap = val_idx.min() - train_idx.max() - 1
    temporal_rows.append(
        (fold, train_idx.min(), train_idx.max(), val_idx.min(), val_idx.max(), observed_gap)
    )
    assert train_idx.max() < val_idx.min()
    assert observed_gap == 2
    assert np.all(np.diff(train_idx) == 1) and np.all(np.diff(val_idx) == 1)

print("fold | treino       | validação    | gap")
for fold, tr0, tr1, va0, va1, gap in temporal_rows:
    print(f"{fold:4d} | {tr0:02d}..{tr1:02d}       | {va0:02d}..{va1:02d}       | {gap}")

### 3.1 Mapa dos índices

Azul indica treino, cinza é o gap e laranja é validação. A janela de treino cresce; a validação sempre avança no tempo.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
for row, (train_idx, val_idx) in enumerate(temporal_cv.split(time_index)):
    ax.scatter(train_idx, np.full_like(train_idx, row), marker="s", s=36,
               color="#2563eb", label="treino" if row == 0 else None)
    gap_idx = np.arange(train_idx.max() + 1, val_idx.min())
    ax.scatter(gap_idx, np.full_like(gap_idx, row), marker="s", s=36,
               color="#9ca3af", label="gap" if row == 0 else None)
    ax.scatter(val_idx, np.full_like(val_idx, row), marker="s", s=36,
               color="#f97316", label="validação" if row == 0 else None)
ax.set(xlabel="Posição temporal", ylabel="Fold",
       yticks=np.arange(4), yticklabels=np.arange(1, 5),
       title="TimeSeriesSplit: passado → gap → futuro")
ax.legend(ncol=3, loc="upper left")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Fluxo iid com teste externo lacrado

Este experimento é separado do caso agrupado. Geramos observações aproximadamente iid, reservamos 20% como teste e aplicamos CV somente ao desenvolvimento. O scaler permanece dentro do pipeline.


In [ ]:
X_iid, y_iid = make_classification(
    n_samples=1500,
    n_features=12,
    n_informative=7,
    n_redundant=2,
    weights=[0.68, 0.32],
    class_sep=1.0,
    flip_y=0.025,
    random_state=SEED + 10,
)
idx = np.arange(len(y_iid))
idx_dev, idx_test = train_test_split(
    idx, test_size=0.20, stratify=y_iid, random_state=SEED + 11
)
X_dev, y_dev = X_iid[idx_dev], y_iid[idx_dev]
X_test, y_test = X_iid[idx_test], y_iid[idx_test]

pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=3000, random_state=SEED),
)
iid_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 12)
print("Desenvolvimento:", X_dev.shape, "Teste lacrado:", X_test.shape)
assert set(idx_dev).isdisjoint(idx_test)

### 4.1 Scores por fold

Calculamos ROC-AUC, log-loss e F1. O cross_validate recria o pipeline em cada rodada. A log-loss é negada pela convenção de scorers do scikit-learn, então revertê-la torna “menor é melhor”.


In [ ]:
iid_res = cross_validate(
    pipeline,
    X_dev,
    y_dev,
    cv=iid_cv,
    scoring={"auc": "roc_auc", "neg_log_loss": "neg_log_loss", "f1": "f1"},
)
auc_folds = iid_res["test_auc"]
logloss_folds = -iid_res["test_neg_log_loss"]
f1_folds = iid_res["test_f1"]

print("ROC-AUC por fold:", np.round(auc_folds, 6))
print(f"ROC-AUC média ± desvio: {auc_folds.mean():.6f} ± {auc_folds.std(ddof=1):.6f}")
print(f"Log-loss média: {logloss_folds.mean():.6f}")
print(f"F1 média: {f1_folds.mean():.6f}")
assert np.all((auc_folds > 0.5) & (auc_folds <= 1))
assert np.all(logloss_folds > 0)

### 4.2 Uma predição OOF por linha

Com folds disjuntos, cada linha recebe probabilidade de um modelo que não a treinou. Recalculamos métricas no vetor OOF e comparamos F1 pooled à média de F1 dos folds.


In [ ]:
oof_probability = cross_val_predict(
    pipeline, X_dev, y_dev, cv=iid_cv, method="predict_proba"
)[:, 1]
oof_prediction = (oof_probability >= 0.5).astype(int)

oof_auc = roc_auc_score(y_dev, oof_probability)
oof_logloss = log_loss(y_dev, oof_probability)
oof_brier = brier_score_loss(y_dev, oof_probability)
oof_f1 = f1_score(y_dev, oof_prediction)

print(f"OOF ROC-AUC: {oof_auc:.6f}")
print(f"OOF log-loss: {oof_logloss:.6f}")
print(f"OOF Brier: {oof_brier:.6f}")
print(f"F1 médio dos folds: {f1_folds.mean():.6f}")
print(f"F1 pooled OOF:      {oof_f1:.6f}")
print(f"Diferença absoluta: {abs(f1_folds.mean() - oof_f1):.6f}")
assert len(oof_probability) == len(y_dev)
assert np.isfinite(oof_probability).all()
assert np.all((oof_probability >= 0) & (oof_probability <= 1))

### 4.3 Refit e avaliação final única

Depois de congelar pipeline, splitter e métricas, ajustamos no desenvolvimento completo e consultamos o teste uma vez. Compatibilidade com CV não significa igualdade exata.


In [ ]:
final_model = pipeline.fit(X_dev, y_dev)
test_probability = final_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

test_auc = roc_auc_score(y_test, test_probability)
test_logloss = log_loss(y_test, test_probability)
test_f1 = f1_score(y_test, test_prediction)

print(f"CV ROC-AUC:    {auc_folds.mean():.6f} ± {auc_folds.std(ddof=1):.6f}")
print(f"Teste ROC-AUC: {test_auc:.6f}")
print(f"Teste log-loss:{test_logloss:.6f}")
print(f"Teste F1:      {test_f1:.6f}")
assert abs(test_auc - auc_folds.mean()) < 0.12
assert test_auc > 0.75

## 5. Verificações consolidadas

As verificações abaixo transformam as hipóteses metodológicas em condições executáveis.


In [ ]:
checks = {
    "seed fixa": SEED == 20260908,
    "otimismo detectado": otimismo > 0.20,
    "grupos disjuntos": bool(np.all(overlap_group == 0)),
    "gap temporal exato": all(row[-1] == 2 for row in temporal_rows),
    "OOF completo": len(oof_probability) == len(y_dev),
    "teste reservado": set(idx_dev).isdisjoint(idx_test),
}
for name, passed in checks.items():
    print(f"{name:24s}: {'OK' if passed else 'FALHOU'}")
assert all(checks.values())

## Conclusões sustentadas

- O split de registros responde a “novos registros de entidades conhecidas”; GroupKFold responde a “entidades novas”.
- A ausência do identificador explícito nas features não impede vazamento de identidade por assinaturas correlacionadas.
- TimeSeriesSplit com gap codifica causalidade e latência, mas o valor do gap ainda precisa vir do domínio.
- OOF permite diagnósticos e agregação; continua sendo desenvolvimento e não substitui o teste.
- A média e a dispersão dos folds devem acompanhar a hipótese do splitter.

**Não sustentado:** estes dados sintéticos não provam qual splitter serve para outro domínio nem fornecem um intervalo de confiança universal.


## Desafio

Altere uma decisão por vez:

1. aumente o ruído das assinaturas das entidades;
2. reduza o número de entidades mantendo o total de registros;
3. troque KNN por regressão logística;
4. mude o gap temporal para refletir outra latência;
5. compare F1 médio, ponderado e pooled quando os folds tiverem tamanhos diferentes.

Antes de executar, escreva a direção esperada da mudança. Registre scores, grupos, índices e conclusão.


## Referências

- [scikit-learn — Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [scikit-learn — GroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)
- [scikit-learn — TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)
- [Cawley e Talbot (2010), JMLR](https://jmlr.org/papers/v11/cawley10a.html)
